In [1]:
%%capture
%pip install -U \
langchain \
langchain-core \
langchain-community \
langchain-text-splitters \
langchain-huggingface \
langchain-chroma \
langchain-classic \
langchain-ibm \
ibm-watsonx-ai \
chromadb \
sentence-transformers \
transformers \
huggingface-hub \
wget

In [2]:
%pip install --upgrade \
numpy \
pandas \
scipy \
scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 856.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 55.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.3
    Uninstalling numpy-2.1.3:
      Successfully uninstalled numpy-2.1.3
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully uninstalled pandas-2.2.3
  Attempting uninstall: scikit-learn
    Found existing installation: scik

In [1]:
def warn(*args, **kwargs):
    pass

import warnings
warnings.warn = warn
warnings.filterwarnings("ignore")

import wget

# LangChain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains import RetrievalQA, ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate

# IBM watsonx
from ibm_watsonx_ai.foundation_models import Model
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import (
    ModelTypes,
    DecodingMethods,
)
from langchain_ibm import WatsonxLLM

print("All imports successful!")

All imports successful!


In [2]:
filename = 'companyPolicies.txt'
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt'

# Use wget to download the file
wget.download(url, out=filename)
print('file downloaded')

file downloaded


In [3]:
with open(filename, 'r') as file:
    # Read the contents of the file
    contents = file.read()
    print(contents)

1.	Code of Conduct

Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.
Integrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.
Respect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.
Accountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we strive to continuously improve our practices. We report any potential violations of 

In [4]:
loader = TextLoader(filename)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)
print(len(texts))

16


In [5]:
embeddings = HuggingFaceEmbeddings()
docsearch = Chroma.from_documents(texts, embeddings)  # store the embedding in docsearch using Chromadb
print('document ingested')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

document ingested


In [6]:
model_id  = 'meta-llama/llama-4-maverick-17b-128e-instruct-fp8'

In [7]:
parameters = {
    GenParams.DECODING_METHOD: DecodingMethods.GREEDY,
    GenParams.MIN_NEW_TOKENS: 130, # this controls the minimum number of tokens in the generated output
    GenParams.MAX_NEW_TOKENS: 256,  # this controls the maximum number of tokens in the generated output
    GenParams.TEMPERATURE: 0.5 # this randomness or creativity of the model's responses
}

In [8]:
credentials = {
    "url": "https://us-south.ml.cloud.ibm.com"
}

project_id = "skills-network"

In [9]:
llm = WatsonxLLM(
    model_id=model_id,
    url="https://us-south.ml.cloud.ibm.com",
    project_id=project_id,
    params=parameters,
)

ValidationError: 1 validation error for WatsonxLLM
  Value error, Did not find 'api_key' or 'token', please add an environment variable `WATSONX_API_KEY` or 'WATSONX_TOKEN' which contains it, or pass 'api_key' or 'token' as a named parameter. [type=value_error, input_value={'model_id': 'meta-llama/...56, 'temperature': 0.5}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

In [10]:
from google.colab import userdata
import os
os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
print(os.environ['GOOGLE_API_KEY'])

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI
MODEL = "gemini-3.5-flash"
llm = ChatGoogleGenerativeAI( model=MODEL, temperature=0.2 )

ModuleNotFoundError: No module named 'langchain_google_genai'

In [12]:
%pip install -qU \ langchain \ langgraph \ langchain-google-genai \ langchain-community \ langchain-text-splitters \ langchain-chroma \ chromadb \ pypdf \ beautifulsoup4


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 732.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 13.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34

In [14]:
from langchain_google_genai import ChatGoogleGenerativeAI
MODEL = "gemini-3.5-flash"
llm = ChatGoogleGenerativeAI( model=MODEL, temperature=0.2 )

In [15]:
qa = RetrievalQA.from_chain_type(llm=llm,
                                 chain_type="stuff",
                                 retriever=docsearch.as_retriever(),
                                 return_source_documents=False)
query = "what is mobile policy?"
qa.invoke(query)

{'query': 'what is mobile policy?',
 'result': 'Based on the provided document, the **Mobile Phone Policy** sets the standards and expectations for the appropriate and responsible use of mobile devices within the organization. Its purpose is to ensure employees use mobile phones in a way that aligns with company values and legal compliance. \n\nThe key guidelines of the policy include:\n\n*   **Acceptable Use:** Mobile devices are primarily for work-related tasks, though limited personal use is allowed if it does not disrupt work.\n*   **Security:** Employees must safeguard their devices and credentials, be cautious when downloading apps or clicking links, and report any security concerns.\n*   **Confidentiality:** Sensitive company information should not be sent via unsecured messaging apps or emails, and employees must be discreet when discussing company matters in public.\n*   **Cost Management:** Personal phone usage must be kept separate from company accounts, and employees must r

In [16]:
qa = RetrievalQA.from_chain_type(llm=llm,
                                 chain_type="stuff",
                                 retriever=docsearch.as_retriever(),
                                 return_source_documents=False)
query = "Can you summarize the document for me?"
qa.invoke(query)

{'query': 'Can you summarize the document for me?',
 'result': "Based on the provided document, here is a summary of the key sections:\n\n*   **Code of Conduct:** Outlines the organization's core ethical standards and values:\n    *   *Integrity:* Acting honestly, protecting sensitive information, and avoiding conflicts of interest.\n    *   *Respect:* Embracing diversity and maintaining a zero-tolerance policy for discrimination or harassment.\n    *   *Accountability:* Taking responsibility for actions, complying with laws, and reporting violations.\n    *   *Safety:* Prioritizing the well-being of employees, clients, and communities.\n    *   *Environmental Responsibility:* Committing to minimizing the organization's environmental footprint.\n*   **Health and Safety Policy (Section 7):** Emphasizes a commitment to maintaining a hazard-free workplace to prevent accidents, injuries, and illnesses. It highlights compliance with safety laws, regular safety assessments, employee training

In [17]:
qa = RetrievalQA.from_chain_type(llm=llm,
                                 chain_type="stuff",
                                 retriever=docsearch.as_retriever(),
                                 return_source_documents=False)
query = "Can I eat in company vehicles?"
qa.invoke(query)

{'query': 'Can I eat in company vehicles?',
 'result': 'Based on the provided policy text, there is no mention of whether or not you can eat in company vehicles. \n\nThe policy only explicitly states that **smoking** (including electronic cigarettes and vaping) is not permitted in company vehicles to maintain their condition and cleanliness.'}

In [18]:
prompt_template = """Use the information from the document to answer the question at the end. If you don't know the answer, just say that you don't know, definitely do not try to make up an answer.

{context}

Question: {question}
"""

PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

chain_type_kwargs = {"prompt": PROMPT}

In [19]:
qa = RetrievalQA.from_chain_type(llm=llm,
                                 chain_type="stuff",
                                 retriever=docsearch.as_retriever(),
                                 chain_type_kwargs=chain_type_kwargs,
                                 return_source_documents=False)

query = "Can I eat in company vehicles?"
qa.invoke(query)

{'query': 'Can I eat in company vehicles?',
 'result': "Based on the provided document, I don't know. The text mentions that smoking is not permitted in company vehicles, but it does not contain any information regarding whether or not you can eat in them."}

In [20]:
memory = ConversationBufferMemory(memory_key = "chat_history", return_message = True)

In [21]:
qa = ConversationalRetrievalChain.from_llm(llm=llm,
                                           chain_type="stuff",
                                           retriever=docsearch.as_retriever(),
                                           memory = memory,
                                           get_chat_history=lambda h : h,
                                           return_source_documents=False)

In [22]:
history = []

In [23]:
query = "What is mobile policy?"
result = qa.invoke({"question":query}, {"chat_history": history})
print(result["answer"])

Based on the provided text, the **Mobile Phone Policy** sets the standards and expectations for the appropriate and responsible use of mobile devices within the organization. 

Its purpose is to ensure that employees use mobile phones in a manner that aligns with company values and legal compliance. The policy covers several key areas:
* **Acceptable Use:** Mobile devices are primarily for work-related tasks, though limited personal use is allowed if it doesn't disrupt work.
* **Security:** Employees must safeguard their devices and credentials, be cautious when downloading apps or clicking links, and report security concerns.
* **Confidentiality:** Sensitive company information should not be sent via unsecured messaging apps or emails, and employees must be discreet in public spaces.
* **Cost Management:** Personal usage must be kept separate from company accounts, and employees must reimburse the company for personal charges on company-issued phones.
* **Compliance:** Employees must 

In [24]:
history.append((query, result["answer"]))

In [25]:
query = "List points in it?"
result = qa({"question": query}, {"chat_history": history})
print(result["answer"])

Based on the provided document, the key points of the Mobile Phone Policy are:

*   **Acceptable Use:** Mobile devices are primarily for work-related tasks. Limited personal use is allowed as long as it does not disrupt work obligations.
*   **Security:** Employees must safeguard their devices and access credentials, exercise caution when downloading apps or clicking links, and promptly report any security concerns.
*   **Confidentiality:** Sensitive company information should not be transmitted via unsecured messaging apps or emails, and employees must be discreet when discussing company matters in public.
*   **Cost Management:** Personal phone usage must be kept separate from company accounts, and employees must reimburse the company for any personal charges incurred on company-issued phones.
*   **Compliance:** Employees must adhere to all relevant laws and regulations, particularly those regarding data protection and privacy.
*   **Lost or Stolen Devices:** Any lost or stolen mobi

In [26]:
history.append((query, result["answer"]))

In [27]:
query = "What is the aim of it?"
result = qa({"question": query}, {"chat_history": history})
print(result)
print(result["answer"])

{'answer': 'Based on the provided document, the Mobile Phone Policy is aimed at promoting the responsible and secure use of mobile devices in line with legal and ethical standards. Its purpose is to ensure that employees utilize mobile phones in a manner consistent with company values and legal compliance.'}
Based on the provided document, the Mobile Phone Policy is aimed at promoting the responsible and secure use of mobile devices in line with legal and ethical standards. Its purpose is to ensure that employees utilize mobile phones in a manner consistent with company values and legal compliance.
